Experiment modelling factors of alignment.

In [1]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from load_results import load, format
from scipy.stats import chi2 as chi2_dist
import itertools
import os
import re
from schedule_similarity import SCRIPTS, KS, NORMALIZE
from schedule_representations import experiments
from plotting_config import scores_to_plot
import warnings
import pandas as pd
from load_results import load, format, abbrev_map_score,abbrev_map_model,abbrev_map_task
warnings.filterwarnings(
    "ignore",
    message=".*incompatible dtype.*", # for annotating significance as *** strings
    category=FutureWarning
)


df = load()

In [2]:
df = df[df["score_name"].isin(scores_to_plot)]
df = format(df)
df = df.drop(columns=["selection_strategy_name_a", "selection_strategy_name_b"])

In [3]:
assert len(df[df["score"].isna()]) == 0


In [4]:
long_df = df[["score_name", "base_model", "Experiment", "train_dataset", "Rep_A", "Rep_B", "test_type", "k", "normalize", "score"]].copy().rename(columns={"score_name": "metric", "train_dataset": "dataset"})
long_df["Rep_pair"] = long_df.apply(
    lambda r: ":".join(sorted([str(r["Rep_A"]), str(r["Rep_B"])])),
    axis=1
)
long_df = long_df.drop(columns=["Rep_A", "Rep_B"])
assert len(long_df[long_df.isna().any(axis=1)]) == 0

In [5]:
df = long_df[
    (long_df["metric"] == "pw_acc") &
    (long_df["k"] == -1) &
    (long_df["normalize"] == False)
]

$y_{i,m,d}=\beta_0+\beta^{(m)}_{m}+\beta^{(d)}_{d}+\beta^{(m \times d)}_{m,d}+u_i+\epsilon_{i,m,d}$

In [6]:
def fit_model(definition, long_df, score_name, k, normalize ):
    if normalize:
        return None
    df = long_df[(long_df["metric"] == score_name) & (long_df["k"] == k)  & (long_df["normalize"] == normalize)].copy()
    
    cols = ["base_model", "dataset", "score", "Rep_pair"]
    nan_rows = df[df[cols].isnull().any(axis=1)]
    
    if df[cols].isnull().values.any():
        print(f"missing entries for {score_name} at k={k}")
        display(nan_rows)
        print(len(df),len(nan_rows))
        return None
    if (df[cols].nunique() == 1).any():
        print(f"constant entries for {score_name} at k={k}")
        return None
    data = df[cols].reset_index(drop=True)

    return smf.mixedlm(
                definition,
                data=data,
                groups=data["Rep_pair"]
            ).fit(reml=False)

configs = long_df.groupby(["metric", "k", "test_type", "normalize"]).groups.keys()
models = {
    (abbrev_map_score[score_name], k, test_type, normalize) : fit_model("score ~ C(base_model) * C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize in configs
} 

models_no_interaction = {
    (abbrev_map_score[score_name], k, test_type, normalize) : fit_model("score ~ C(base_model) + C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_base_model = {
    (abbrev_map_score[score_name], k, test_type, normalize) : fit_model("score ~ C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_dataset = {
    (abbrev_map_score[score_name], k, test_type, normalize) : fit_model("score ~ C(base_model)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 


In [7]:
def lr_test_df(full_model, reduced_model, term_name):
    lr_stat = 2 * (full_model.llf - reduced_model.llf)

    df_diff = full_model.df_modelwc - reduced_model.df_modelwc
    p_value = chi2_dist.sf(lr_stat, df_diff)

    return pd.DataFrame({
        term_name: [term_name],
        "chi2": [lr_stat],
        "P>chi2": [p_value],
        "df constraint": [df_diff]
    }).set_index(term_name)
def compute_r2_mixedlm(model):
    # fixed effects design matrix
    fe = model.model.exog
    fe_params = model.fe_params.values

    # fixed-effect linear predictor η
    eta = fe @ fe_params
    sigma_f2 = np.var(eta, ddof=1)

    # random intercept variance (Rep_pair)
    sigma_pair2 = model.cov_re.values.item()

    # residual variance
    sigma_e2 = model.scale

    denom = sigma_f2 + sigma_pair2 + sigma_e2

    R2_m = sigma_f2 / denom
    R2_c = (sigma_f2 + sigma_pair2) / denom

    return sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c


result_dfs_lrt = []
result_dfs_r2 = []
for (score_name, k, test_type, normalize), model in models.items():
    if model is None:
        continue
    interaction_test = pd.concat([
        lr_test_df(
            model,
            models_no_base_model[(score_name, k, test_type, normalize)],
            "C(base_model)"
        ),
        lr_test_df(
            model,
            models_no_dataset[(score_name, k, test_type,normalize)],
            "C(dataset)"
        ),
        lr_test_df(
            model,
            models_no_interaction[(score_name, k, test_type,normalize)],
            "C(base_model):C(dataset)"
        )
        ])
    df_model_dataset = interaction_test.stack().rename_axis(["term", "stat"]).to_frame().T.reset_index(drop=True)
    df_model_dataset["metric"] = score_name
    df_model_dataset["test_type"] = test_type
    df_model_dataset["k"] = k
    df_model_dataset["normalize"] = normalize
    df_model_dataset = df_model_dataset.set_index(["metric", "k", "normalize"])
    result_dfs_lrt.append(df_model_dataset)
    
    r2_dfs = []
    for setting, model_ in [
        ("full", model),
        ("no_interaction", models_no_interaction[(score_name, k,test_type, normalize)]),
        ("no_base_model", models_no_base_model[(score_name, k,test_type,normalize)]),
        ("no_dataset", models_no_dataset[(score_name, k,test_type,normalize)])
        ]:
        sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c = compute_r2_mixedlm(model_)
        df_r2 = pd.DataFrame({

            "metric": [score_name],
            "test_type": [test_type],
            "normalize": [normalize],
            "k": [k],
            "R2_m": [R2_m],
            "R2_c": [R2_c]
        })
        df_r2.columns = pd.MultiIndex.from_product(
            [[setting], df_r2.columns]
        )

        r2_dfs.append(df_r2)
    result_dfs_r2.append(pd.concat(r2_dfs, axis=1))
        
 
df_lrt = pd.concat(result_dfs_lrt)
df_r2_model_dataset = pd.concat(result_dfs_r2)


# LRTs

In [8]:
df_display = df_lrt.reset_index().set_index(
    ["test_type", "metric", "k"]
).sort_values(["test_type", "metric", "k"])

df_display = df_display.loc[
    :, df_display.columns.get_level_values("stat") != "df constraint"
]
df_display.index = df_display.index.set_names(
    ["Test type", "Metric \\rotatebox[origin=c]{270}{$\Rsh$}", "k"]
)
df_display = df_display.rename(columns={
    "C(base_model)": "Base model",
    "C(dataset)": "Dataset",
    "C(base_model):C(dataset)": "Interaction"
})
df_display.columns = df_display.columns.set_names(["Term $\\rightarrow$", "stat"])


chi2_df = df_display.xs("chi2", level="stat", axis=1)
p_df = df_display.xs("P>chi2", level="stat", axis=1)


def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

stars_df = p_df.map(p_to_star)


display_df = df_display.copy()

mask_p = display_df.columns.get_level_values("stat") == "P>chi2"


display_df.loc[:, mask_p] = stars_df.values


mask_chi2 = display_df.columns.get_level_values("stat") == "chi2"


styled = (
    display_df.style
    .background_gradient(
        cmap="viridis",
        subset=display_df.columns[mask_chi2],
        axis=1
    )
    .hide(axis="columns", level="stat")
    .hide(axis="columns", subset=[ "normalize"])
    .hide(axis="index", level=[0,2])
).format_index(escape="latex", axis=1)\
    .format(precision=1) 
display(styled)
latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="l",
                                   column_format='l' + 'l' * len(styled.data.columns))
with open("figures/lrt-table.tex", "w", encoding="utf-8") as f:
            f.write(latex)
            


In [9]:
def write_raw_latex(row, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    score = re.sub(r"[^a-zA-Z0-9]+", "", str(row.name))
    file_path = os.path.join(out_dir, f"{score}.tex")

    # raw LaTeX content (just the number)
    with open(file_path, "w") as f:
        f.write(f'{row["chi2"]:.2f}{row["P>chi2"]}')

    return None  

styled.data["Base model"].apply(lambda row : write_raw_latex(row, out_dir = "latex_commands/lrt/base-model"), axis=1)
""

''

## $\Delta R^2$

In [10]:
df = df_r2_model_dataset.copy()

df = df.set_index(("full", "metric"))

delta_r2 = pd.DataFrame(index=df.index)

delta_r2["interaction_model_dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_interaction", "R2_m")]
)
delta_r2["interaction_model_dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_interaction", "R2_c")]
)

delta_r2["base_model_R2_m"] = (
    df[("full", "R2_m")] - df[("no_base_model", "R2_m")]
)
delta_r2["base_model_R2_c"] = (
    df[("full", "R2_c")] - df[("no_base_model", "R2_c")]
)

delta_r2["dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_dataset", "R2_m")]
)
delta_r2["dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_dataset", "R2_c")]
)

In [11]:
import pandas as pd

delta_r2_table = pd.DataFrame({
    "$\Delta R_{m}^{2}(\mathcal{M}_{-B})$": delta_r2["base_model_R2_m"],
    "$\Delta R_{c}^{2}(\mathcal{M}_{-B})$": delta_r2["base_model_R2_c"],

    "$\Delta R_{m}^{2}(\mathcal{M}_{-D})$": delta_r2["dataset_R2_m"],
    "$\Delta R_{c}^{2}(\mathcal{M}_{-D})$": delta_r2["dataset_R2_c"],

    "$\Delta R_{m}^{2}(\mathcal{M}_{add})$": delta_r2["interaction_model_dataset_R2_m"],
    "$\Delta R_{c}^{2}(\mathcal{M}_{add})$": delta_r2["interaction_model_dataset_R2_c"],
})
delta_r2_table.index.name = None


delta_r2_table.columns = delta_r2_table.columns.set_names(["Term $\\rightarrow$"])
styled = (
    delta_r2_table.style
    .background_gradient(
        cmap="viridis",
        axis=1  
    )
    .format("{:.2e}")
)
display(styled)

latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="l",
                                   column_format='l' + 'l' * len(styled.data.columns))
with open("figures/delta-r2-table.tex", "w", encoding="utf-8") as f:
            f.write(latex)

Term $\rightarrow$,$\Delta R_{m}^{2}(\mathcal{M}_{-B})$,$\Delta R_{c}^{2}(\mathcal{M}_{-B})$,$\Delta R_{m}^{2}(\mathcal{M}_{-D})$,$\Delta R_{c}^{2}(\mathcal{M}_{-D})$,$\Delta R_{m}^{2}(\mathcal{M}_{add})$,$\Delta R_{c}^{2}(\mathcal{M}_{add})$
CKA,9.88e-03,1.01e-02,6.89e-03,7.07e-03,2.25e-03,2.31e-03
CKNNA@25,4.64e-03,4.77e-03,1.33e-02,1.36e-02,1.59e-03,1.64e-03
$\tau$,5.16e-05,5.30e-05,1.21e-04,1.24e-04,1.29e-05,1.32e-05
$acc_{\text{pred}}$,6.74e-03,6.92e-03,7.67e-03,7.87e-03,3.72e-03,3.82e-03
RBO,1.04e-04,1.05e-04,2.89e-04,2.94e-04,4.95e-05,5.02e-05
$\rho$,9.74e-05,9.96e-05,2.07e-04,2.12e-04,2.40e-05,2.45e-05


### 